[Guide Index](README.md)

# 04b SRL: more features

More of what rules can do: triple terms and annotations (RDF 1.2), recording which rule derived a triple, property paths, grouping and aggregation, and combining SRL with entailment and SHACL. [04](04-srl-sparql-rules.ipynb) shows how to write rules and [04a](04a-srl-running-rule-sets.ipynb) explains how a rule set runs. The examples use a software-dependency graph.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import NamedNode, StarLayerGraph
from starlayer.sparql import srl

EX = "http://example.org/"


def ex(name):
    """A term in the example namespace, as a pyoxigraph NamedNode."""
    return NamedNode(EX + name)


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses.
    (A new StarLayerGraph has no prefixes; the ones used here are bound explicitly.)"""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 1. The base graph

The examples use a small software-dependency graph, the same running example as [the SRL spec's worked example](https://www.w3.org/TR/sparql12-rl/#worked-example): a `:frontend` depends on an `:app`, which depends on a `:db` that has a known vulnerability. A `:logger` component has no vulnerabilities at all.

In [2]:
base = StarLayerGraph(prefixes={"ex": EX})
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .  # no vulnerability - stays safe in section 4

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")


print("data graph")
show(base)

data graph
@prefix ex: <http://example.org/> .

ex:app a ex:Component ;
    ex:dependsOn ex:db .

ex:db a ex:Component ;
    ex:hasVulnerability ex:vuln1 .

ex:frontend a ex:Component ;
    ex:dependsOn ex:app .

ex:logger a ex:Component .

ex:vuln1 ex:severity 9.1 .



## 2. Triple terms and annotations (RDF 1.2)

A **triple term** `<<( s p o )>>` is a whole triple used as a value, so a rule can talk about a claim without asserting it. Rule patterns, rule heads and `DATA` blocks can all contain them, and the variables inside one are ordinary rule variables. Rules can also use the annotation syntax of SPARQL 1.2, shown below.

Here Alice believes that Bob is 130 years old and Carol believes that Dan is 40. The rule matches the claim inside the triple term, with `?person` and `?age` bound from within it, and records who holds an implausible belief:

In [3]:
claims = StarLayerGraph(prefixes={"ex": EX})
claims.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:believes <<( ex:bob ex:age 130 )>> .
    ex:carol ex:believes <<( ex:dan ex:age 40 )>> .
""", format="turtle")

doubt_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?who ex:doubts ?person }
WHERE { ?who ex:believes <<( ?person ex:age ?age )>> . FILTER(?age > 120) }
""")

doubts = doubt_rules.infer(claims)
doubts.bind("ex", EX)
show(doubts)

assert (ex("alice"), ex("doubts"), ex("bob")) in doubts
assert len(doubts) == 1

@prefix ex: <http://example.org/> .

ex:alice ex:doubts ex:bob .



A rule **head** can build a triple term too. The variables in the head's triple term are filled in from the rule body. The rule below records that a review disputes each implausible claim: it builds the claim itself, `<<( ex:bob ex:age 130 )>>`, as the object of a new triple, without asserting that Bob is 130:

In [4]:
from starlayer.graph import Literal, Triple

dispute_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ex:review ex:disputes <<( ?person ex:age ?age )>> }
WHERE { ?who ex:believes <<( ?person ex:age ?age )>> . FILTER(?age > 120) }
""")

disputes = dispute_rules.infer(claims)
disputes.bind("ex", EX)
show(disputes)

# a triple term is a pyoxigraph Triple; 130 is an xsd:integer literal
age130 = Literal("130", datatype=NamedNode("http://www.w3.org/2001/XMLSchema#integer"))
assert (ex("review"), ex("disputes"), Triple(ex("bob"), ex("age"), age130)) in disputes
assert len(disputes) == 1

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:review ex:disputes <<( ex:bob ex:age 130 )>> .



Various ways of writing a triple that has been annotated work as they do in SPARQL 1.2: an **annotation block** `{| ... |}`, a **reifier** `~ ?r`, and a **reified triple** `<< s p o >>`. Each one stands for a triple, a reifier for it, and the statements made about the reifier.

In this example the claims carry a certainty. The rule keeps only those at least half certain, matching the certainty through the annotation block:

In [5]:
annotated = StarLayerGraph(prefixes={"ex": EX})
annotated.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:says ex:bobIsOld {| ex:certainty 0.9 |} .
    ex:alice ex:says ex:bobIsTall {| ex:certainty 0.2 |} .
""", format="turtle")

strong_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?who ex:strongClaim ?claim }
WHERE { ?who ex:says ?claim {| ex:certainty ?c |} . FILTER(?c > 0.5) }
""")

strong = strong_rules.infer(annotated)
strong.bind("ex", EX)
show(strong)

assert (ex("alice"), ex("strongClaim"), ex("bobIsOld")) in strong
assert len(strong) == 1

@prefix ex: <http://example.org/> .

ex:alice ex:strongClaim ex:bobIsOld .



### Recording which rule derived a triple

A rule head can use the annotation syntax too, so a rule can say where a triple came from. A rule can be given an id after `RULE`, and the id is an IRI that you can write in the head. StarLayer does not add it for you: you repeat the id in the head, as below. A named reifier (`~ ex:stmt1`) works the same way, and so does a separate head triple that uses a triple term, such as `ex:R1 ex:derived <<( ?x ex:exposedTo ?v )>>`.

In [6]:
provenance_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE ex:R1 { ?x ex:exposedTo ?v {| ex:derivedBy ex:R1 |} } WHERE { ?x ex:hasVulnerability ?v }
""")

derived = provenance_rules.infer(base)
derived.bind("ex", EX)
show(derived)

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:db ex:exposedTo ex:vuln1 {| ex:derivedBy ex:R1 |} .



## 3. Property paths

A rule body can use a **property path** as a predicate, to follow several steps in one pattern. SRL allows just two path operators:
- `a/b` follows `a` and then `b`;
- `^a` follows `a` backwards.

Parentheses group them, as in `^(a/b)`. Paths are for rule bodies, including inside a `NOT`. A rule head and a `DATA` block take plain predicates only.

Using the dependency graph from section 1, the first rule finds a component that reaches a vulnerability in two steps, and the second turns a vulnerability back into the component that has it:

In [7]:
# `base` is the software-dependency graph from section 1
path_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:affectedVia ?v } WHERE { ?x ex:dependsOn/ex:hasVulnerability ?v }
RULE { ?v ex:affects ?x }     WHERE { ?v ^ex:hasVulnerability ?x }
""")

path_inferred = path_rules.infer(base)
path_inferred.bind("ex", EX)
show(path_inferred)

assert (ex("app"), ex("affectedVia"), ex("vuln1")) in path_inferred   # app depends on db, which has vuln1
assert (ex("vuln1"), ex("affects"), ex("db")) in path_inferred

@prefix ex: <http://example.org/> .

ex:app ex:affectedVia ex:vuln1 .

ex:vuln1 ex:affects ex:db .



Any other path operator (`|`, `*`, `+`, `?`, `!`) is rejected when the rule set is parsed. The spec leaves them out on purpose: a step that repeats is written as a recursive rule instead, like `ex:ancestorOf` in [04](04-srl-sparql-rules.ipynb), and an alternative is written as two rules.

In [8]:
for path in ("ex:dependsOn+", "ex:dependsOn|ex:hasVulnerability"):
    try:
        srl.parse_ruleset(f"""
        PREFIX ex: <http://example.org/>
        RULE {{ ?x ex:reaches ?y }} WHERE {{ ?x {path} ?y }}
        """)
    except srl.SRLParseError as e:
        print(f"{path}:", e)

ex:dependsOn+: SRL: property path <http://example.org/dependsOn>+ is not allowed: SRL property paths use only the / and ^ operators (write a repeated or optional step with a recursive rule instead)
ex:dependsOn|ex:hasVulnerability: SRL: property path (<http://example.org/dependsOn>|<http://example.org/hasVulnerability>) is not allowed: SRL property paths use only the / and ^ operators (write a repeated or optional step with a recursive rule instead)


## 4. Grouping and aggregation

A **group element** collects the solutions of a pattern into groups and computes a value for each group: `GROUP BY ?key { pattern } [ ?n := COUNT(*) ]`. The aggregation functions are `COUNT` (also `COUNT(DISTINCT ?v)` and `COUNT(*)`), `MIN`, `MAX` and `SUM`; the argument is a variable, not an expression. To aggregate an expression, compute it first with `SET` inside the group pattern.

How it is evaluated:

- For each solution reaching the group element, the group pattern is matched with **only the group keys** carried in. Other variables bound before the group are not visible inside it.
- The pattern's solutions are grouped by the keys, and each group gives **one** solution: the one that came in, plus the keys and the aggregation variables. The pattern's own variables (`?v` below) are **not** in scope afterwards, so they cannot be used in a later element or in the rule head.
- A key with no pattern solution has no group, so no triple is inferred for it. With **no** `BY` there is always exactly one group, even if the pattern matches nothing: `COUNT` is then 0 and `SUM` is 0, but `MIN` and `MAX` of nothing are errors, and **an error drops the group**.
- A rule with a group element is a **run-once** rule: it waits for the rules that produce its triples to finish (see [04a](04a-srl-running-rule-sets.ipynb)). A rule that groups over its own output cannot be ordered and is rejected with a `StratificationError`.

Count the vulnerabilities each component is exposed to, using the exposure rules:

In [9]:
count_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

# how many vulnerabilities each component is exposed to
RULE { ?x ex:vulnerabilityCount ?n }
WHERE {
    ?x a ex:Component .
    GROUP BY ?x { ?x ex:exposedTo ?v }
    [ ?n := COUNT(*) ]
}
""")

counted = count_rules.infer(base)
counted.bind("ex", EX)
show(counted)

@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:vulnerabilityCount 1 .

ex:db ex:exposedTo ex:vuln1 ;
    ex:vulnerabilityCount 1 .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:vulnerabilityCount 1 .



`ex:logger` is not exposed to anything, so its group has no solution and it gets no `ex:vulnerabilityCount` at all. To count that as 0, add a rule that fills in a default, as in [04a](04a-srl-running-rule-sets.ipynb) (`NOT DATA`).

With no `BY` there is a single group over the whole pattern, which gives totals. A rule can have several group elements; here one counts the components and another finds the highest severity:

In [10]:
totals = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

RULE { ex:report ex:components ?n ; ex:worstSeverity ?top }
WHERE {
    GROUP { ?c a ex:Component }      [ ?n := COUNT(*) ]
    GROUP { ?v ex:severity ?s }      [ ?top := MAX(?s) ]
}
""")

summary = totals.infer(base)
summary.bind("ex", EX)
show(summary)

@prefix ex: <http://example.org/> .

ex:report ex:components 4 ;
    ex:worstSeverity 9.1 .



## 5. Combining SRL with entailment and SHACL

`infer()` and `query()` take `inference=`, an entailment regime that is applied to the base graph before the rules run (see [SPARQL entailment](03c-sparql-entailment.ipynb)). Here `ex:cache` is declared a `Service`, and a `Service` is a kind of `Component`. The rule needs a `Component`, so it only matches `ex:cache` when RDFS entailment is applied first:

In [11]:
from starlayer.graph import ENTAILMENT

services = StarLayerGraph(prefixes={"ex": EX})
services.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Service rdfs:subClassOf ex:Component .
    ex:cache a ex:Service .
""", format="turtle")

component_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:status ex:checked } WHERE { ?x a ex:Component }
""")

print("without inference=:           ", len(component_rules.infer(services)), "triples inferred")
print("with inference=ENTAILMENT.RDFS:", len(component_rules.infer(services, inference=ENTAILMENT.RDFS)), "triples inferred")

without inference=:            0 triples inferred
with inference=ENTAILMENT.RDFS: 1 triples inferred


`infer()` also takes `schema=`, SHACL shapes that the inferred triples are checked against. The result is then a `RulesResult`, with the inferred graph and the validation of it. Here a shape says that a risk score is at most 100. A rule that multiplies the severity by 10 conforms, and one that multiplies by 20 does not:

In [12]:
exposure_text = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }
"""

shapes = StarLayerGraph(prefixes={"ex": EX})
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ScoreShape a sh:NodeShape ;
        sh:targetSubjectsOf ex:riskScore ;
        sh:property [ sh:path ex:riskScore ; sh:maxInclusive 100 ] .
""", format="turtle")

for factor in (10, 20):
    rules = srl.parse_ruleset(exposure_text + f"""
    RULE {{ ?x ex:riskScore ?r }}
    WHERE {{ ?x ex:exposedTo ?v . ?v ex:severity ?s . SET(?r := ?s * {factor}) }}
    """)
    result = rules.infer(base, schema=shapes)
    print(f"score = severity * {factor}: conforms =", result.validation.conforms, f"({len(result.inferred_graph)} triples inferred)")

score = severity * 10: conforms = True (6 triples inferred)
score = severity * 20: conforms = False (6 triples inferred)


## 6. Differences from SPARQL

The patterns, `FILTER` expressions and functions of a rule body are SPARQL's, and a rule body is matched with SPARQL. SRL leaves some parts of SPARQL out, and a rule set that uses one is rejected when it is parsed. The left-out parts are:

- **Graph pattern operators.** A rule body is a list of triple patterns, `FILTER`, `NOT`, `SET` and `GROUP`, so there is no `OPTIONAL`, `UNION`, `MINUS`, `BIND`, `VALUES`, `GRAPH`, `SERVICE` or sub-select. `NOT` replaces `FILTER NOT EXISTS` and `MINUS`, and `SET` replaces `BIND`.
- **Some functions.** `EXISTS`, `NOT EXISTS`, `BOUND`, `COALESCE`, `RAND` and the hash functions such as `SHA256` are not part of SRL's set of built-in functions.
- **Aggregates in an expression.** `COUNT`, `SUM` and the rest work only in a `GROUP` element (section 4).
- **Property path operators.** Only `/` and `^` are accepted (section 3).

This cell parses a rule body for each, to show what is accepted and what the error says:

In [13]:
cases = {
    "FILTER with REGEX, IN and &&": "?x ex:a ?y . FILTER(REGEX(STR(?x), 'a') && ?x IN (ex:a, ex:b))",
    "OPTIONAL": "?x ex:a ?y . OPTIONAL { ?x ex:q ?z }",
    "UNION": "{ ?x ex:a ?y } UNION { ?x ex:b ?y }",
    "BIND": "?x ex:a ?y . BIND(1 AS ?z)",
    "FILTER NOT EXISTS": "?x ex:a ?y . FILTER NOT EXISTS { ?x ex:q ?z }",
    "COALESCE": "?x ex:a ?y . FILTER(COALESCE(?y, 1) > 0)",
    "COUNT in a FILTER": "?x ex:a ?y . FILTER(COUNT(?y) > 1)",
    "path ex:a/ex:b": "?x ex:a/ex:b ?z",
    "path ex:a*": "?x ex:a* ?z",
}

for name, body in cases.items():
    try:
        srl.parse_ruleset(f"PREFIX ex: <http://example.org/> RULE {{ ?x ex:p ex:o }} WHERE {{ {body} }}")
        print(f"{name:32} accepted")
    except srl.SRLParseError as error:
        print(f"{name:32} rejected: {str(error)[:70]}")

FILTER with REGEX, IN and &&     accepted
OPTIONAL                         rejected: SPARQL syntax error at line 1, column 77: expected a triple pattern (f
UNION                            rejected: SPARQL syntax error at line 1, column 64: expected a triple pattern (f
BIND                             rejected: SPARQL syntax error at line 1, column 77: expected a triple pattern (f
FILTER NOT EXISTS                rejected: SRL: NOT EXISTS is not part of SRL's built-in function set (spec Built
COALESCE                         rejected: SRL: COALESCE is not part of SRL's built-in function set (spec BuiltIn
COUNT in a FILTER                rejected: SRL: rule set is not well-formed per §4.2: aggregate_in_expression (in
path ex:a/ex:b                   accepted
path ex:a*                       rejected: SRL: property path <http://example.org/a>* is not allowed: SRL propert


## 7. Not yet covered

These parts of SRL are not in StarLayer yet, or not finished in the specification. This guide will gain a section for each when they are:

- **Aggregation inside an expression.** `COUNT`, `SUM` and the rest are only available in a `GROUP` element (section 4). An aggregate inside `FILTER` or `SET` is rejected when the rule set is parsed.
- **Translating a rule set to SPARQL.** A placeholder appendix in the draft (Issue 1303).
- **Choosing among valid evaluation orders.** A rule set can have more than one valid stratification; `plan()` shows the one StarLayer used.